# LLMOps Assignment 2 — API Cost, Latency, and Prompt Version Management
**Course:** MLOps | **Scenario:** EduAssist AI  
This notebook calls the configured LLM for the same five academic questions under three prompt versions. It records request latency and token-based cost, registers prompts in MLflow, logs aggregate metrics, checks JSON validity for v3, and assigns a `production` alias based on the measured results.

> **Important:** Running the notebook makes real API calls and may incur charges. Set the API key through an environment variable. Do not commit secrets. The notebook intentionally does not include fabricated results; generated CSVs reflect actual runs.

In [ ]:
import os, json, time, statistics, math
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import mlflow
from openai import OpenAI

OUTPUT_DIR = Path(".")
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
INPUT_PRICE_PER_MILLION = float(os.getenv("OPENAI_INPUT_PRICE_PER_MILLION", "0.40"))
OUTPUT_PRICE_PER_MILLION = float(os.getenv("OPENAI_OUTPUT_PRICE_PER_MILLION", "1.60"))
MONTHLY_REQUESTS = 10_000
EXPERIMENT_NAME = "eduassist_llmops_assignment"

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("Set OPENAI_API_KEY in your shell/environment before running this notebook.")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("Model:", MODEL)
print("Pricing (USD per 1M tokens): input =", INPUT_PRICE_PER_MILLION, ", output =", OUTPUT_PRICE_PER_MILLION)
print("MLflow version:", mlflow.__version__)

In [ ]:
questions = [
    {"question_id": "Q1", "question": "What is the difference between supervised and unsupervised learning?"},
    {"question_id": "Q2", "question": "Explain overfitting in machine learning and give two ways to reduce it."},
    {"question_id": "Q3", "question": "What is the purpose of a train-test split when building a machine learning model?"},
    {"question_id": "Q4", "question": "Explain precision and recall, and describe when each metric is useful."},
    {"question_id": "Q5", "question": "What is data drift, and why should a deployed machine learning system monitor it?"}
]

PROMPTS = {
    "v1": "Answer the following student question clearly.\nQuestion: {{question}}",
    "v2": "You are an academic assistant.\nAnswer the following student question in a clear, concise, and accurate manner.\nUse simple language suitable for undergraduate students.\nQuestion: {{question}}",
    "v3": '''You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
 "answer": "string",
 "confidence": "high | medium | low",
 "key_points": ["string"]
}
Question: {{question}}'''
}
COMMIT_MESSAGES = {
    "v1": "Initial baseline academic QA prompt",
    "v2": "Add academic-assistant role and undergraduate-friendly style guidance",
    "v3": "Require schema-constrained JSON output for downstream validation"
}
questions

## Task 1 and Task 2 — API calls, latency, tokens, and costs
We run each prompt version against the same five questions (15 requests total). Token usage is taken from the API response. Failed calls are recorded with a status and error, but their missing token counts are not treated as zero-cost successes.

In [ ]:
def render_prompt(template: str, question: str) -> str:
    return template.replace("{{question}}", question)

def cost_usd(input_tokens, output_tokens):
    return (input_tokens / 1_000_000 * INPUT_PRICE_PER_MILLION
            + output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MILLION)

records = []
raw_outputs = {}
for version, template in PROMPTS.items():
    for item in questions:
        started = datetime.now(timezone.utc)
        t0 = time.perf_counter()
        status, output, input_tokens, output_tokens, error = "success", "", None, None, ""
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": render_prompt(template, item["question"])}],
                temperature=0,
                max_tokens=350,
            )
            output = response.choices[0].message.content or ""
            usage = response.usage
            input_tokens = int(usage.prompt_tokens) if usage else None
            output_tokens = int(usage.completion_tokens) if usage else None
            raw_outputs[(version, item["question_id"])] = output
        except Exception as exc:
            status = "error"
            error = f"{type(exc).__name__}: {exc}"
        elapsed = time.perf_counter() - t0
        ended = datetime.now(timezone.utc)
        records.append({
            "question_id": item["question_id"],
            "prompt_version": version,
            "model_name": MODEL,
            "start_time_utc": started.isoformat(),
            "end_time_utc": ended.isoformat(),
            "latency_seconds": round(elapsed, 6),
            "response_status": status,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": (input_tokens + output_tokens) if input_tokens is not None and output_tokens is not None else None,
            "input_cost_usd": (input_tokens / 1_000_000 * INPUT_PRICE_PER_MILLION) if input_tokens is not None else None,
            "output_cost_usd": (output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MILLION) if output_tokens is not None else None,
            "cost_per_request_usd": cost_usd(input_tokens, output_tokens) if input_tokens is not None and output_tokens is not None else None,
            "error": error,
            "response_text": output
        })

latency_df = pd.DataFrame(records)
latency_df.to_csv("latency_tokens_cost.csv", index=False)
display(latency_df[["question_id","prompt_version","model_name","latency_seconds","response_status","input_tokens","output_tokens","total_tokens","cost_per_request_usd"]])
print("Saved latency_tokens_cost.csv")

In [ ]:
successful = latency_df.loc[latency_df["response_status"] == "success"].copy()
latencies = successful["latency_seconds"].dropna()
latency_summary = pd.DataFrame([{
    "count": int(latencies.count()),
    "mean_seconds": float(latencies.mean()) if len(latencies) else None,
    "median_seconds": float(latencies.median()) if len(latencies) else None,
    "min_seconds": float(latencies.min()) if len(latencies) else None,
    "max_seconds": float(latencies.max()) if len(latencies) else None,
    "std_seconds_sample": float(latencies.std(ddof=1)) if len(latencies) > 1 else 0.0,
}])
display(latency_summary)
if len(successful):
    slowest = successful.loc[successful["latency_seconds"].idxmax()]
    print(f"Slowest successful request: {slowest['question_id']} / {slowest['prompt_version']} at {slowest['latency_seconds']:.3f}s.")
    print("Possible reasons include network/service variability, request queueing, prompt length, and generated output length. One observation alone cannot identify the cause.")

## Task 3 — MLflow prompt registry and evaluation
The notebook registers v1–v3 under one prompt name, logs an MLflow run for each version, and computes the required aggregate metrics. `valid_json` is the fraction of successful responses that parse as JSON with the required keys and value types. For v1/v2, JSON validity is still measured but is not a required output property; the comparison table will show their observed rates.

In [ ]:
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000"))
mlflow.set_experiment(EXPERIMENT_NAME)

PROMPT_NAME = "eduassist_academic_qa"

# Prompt Registry is available in modern MLflow releases. If the server/client
# is older, upgrade MLflow before running this section.
registered = {}
for version, template in PROMPTS.items():
    registered[version] = mlflow.genai.register_prompt(
        name=PROMPT_NAME,
        template=template,
        commit_message=COMMIT_MESSAGES[version],
        tags={"assignment": "llmops_assignment_2", "prompt_version_label": version}
    )
    print(version, "registered as MLflow prompt version", registered[version].version)

In [ ]:
def is_valid_eduassist_json(text):
    try:
        obj = json.loads(text)
        if not isinstance(obj, dict):
            return False
        if not isinstance(obj.get("answer"), str):
            return False
        if obj.get("confidence") not in {"high", "medium", "low"}:
            return False
        if not isinstance(obj.get("key_points"), list) or not all(isinstance(x, str) for x in obj["key_points"]):
            return False
        return True
    except (json.JSONDecodeError, TypeError):
        return False

evaluation_rows = []
for version in PROMPTS:
    subset = latency_df[(latency_df["prompt_version"] == version) & (latency_df["response_status"] == "success")].copy()
    validity = [is_valid_eduassist_json(raw_outputs.get((version, qid), "")) for qid in subset["question_id"]]
    avg_latency = subset["latency_seconds"].mean() if len(subset) else float("nan")
    avg_input = subset["input_tokens"].dropna().mean() if len(subset) else float("nan")
    avg_output = subset["output_tokens"].dropna().mean() if len(subset) else float("nan")
    avg_cost = subset["cost_per_request_usd"].dropna().mean() if len(subset) else float("nan")
    valid_json_rate = (sum(validity) / len(validity)) if validity else 0.0
    evaluation_rows.append({
        "prompt_version": version,
        "mlflow_prompt_version": str(registered[version].version),
        "avg_latency_seconds": avg_latency,
        "avg_input_tokens": avg_input,
        "avg_output_tokens": avg_output,
        "avg_cost_per_request_usd": avg_cost,
        "valid_json": valid_json_rate,
        "successful_responses": int(len(subset)),
        "recommended": False
    })

eval_df = pd.DataFrame(evaluation_rows)
display(eval_df)


In [ ]:
# Log metrics to MLflow. Missing values are omitted rather than logged as NaN.
for _, row in eval_df.iterrows():
    version = row["prompt_version"]
    with mlflow.start_run(run_name=f"prompt_{version}") as run:
        mlflow.log_param("prompt_version_label", version)
        mlflow.log_param("mlflow_prompt_version", row["mlflow_prompt_version"])
        mlflow.log_param("model_name", MODEL)
        mlflow.log_param("input_price_per_million_usd", INPUT_PRICE_PER_MILLION)
        mlflow.log_param("output_price_per_million_usd", OUTPUT_PRICE_PER_MILLION)
        for metric in ["avg_latency_seconds", "avg_input_tokens", "avg_output_tokens", "avg_cost_per_request_usd", "valid_json"]:
            value = row[metric]
            if pd.notna(value):
                mlflow.log_metric(metric, float(value))
        mlflow.log_metric("successful_responses", float(row["successful_responses"]))

# Choose a production candidate transparently: prefer JSON-valid v3 if it achieves
# 100% validity; otherwise choose the highest valid_json rate, then lower cost,
# then lower latency. This rule is deterministic for the observed results.
ranked = eval_df.copy()
ranked["valid_json_sort"] = ranked["valid_json"].fillna(-1)
ranked["cost_sort"] = ranked["avg_cost_per_request_usd"].fillna(float("inf"))
ranked["latency_sort"] = ranked["avg_latency_seconds"].fillna(float("inf"))
ranked = ranked.sort_values(["valid_json_sort", "cost_sort", "latency_sort"], ascending=[False, True, True])
winner = ranked.iloc[0]["prompt_version"]
eval_df.loc[eval_df["prompt_version"] == winner, "recommended"] = True

# Assign the production alias to the selected registered prompt version.
winner_mlflow_version = int(eval_df.loc[eval_df["prompt_version"] == winner, "mlflow_prompt_version"].iloc[0])
mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="production", version=winner_mlflow_version)

eval_df.to_csv("prompt_evaluation_results.csv", index=False)
print(f"Recommended version: {winner}; assigned MLflow alias '{PROMPT_NAME}@production' -> version {winner_mlflow_version}.")
display(eval_df)
print("Saved prompt_evaluation_results.csv")

In [ ]:
# Summary tables for the report
cost_summary = successful.groupby("prompt_version", as_index=False).agg(
    successful_requests=("question_id", "count"),
    avg_cost_per_request_usd=("cost_per_request_usd", "mean"),
    avg_input_tokens=("input_tokens", "mean"),
    avg_output_tokens=("output_tokens", "mean"),
    avg_latency_seconds=("latency_seconds", "mean"),
)
cost_summary["projected_monthly_cost_10000_usd"] = cost_summary["avg_cost_per_request_usd"] * MONTHLY_REQUESTS
display(cost_summary)

print("Monthly projection formula: average cost per successful request × 10,000.")
print("Pricing is an estimate in USD, based on the configured model rates and API usage counts.")

## Required interpretation and screenshots
- **Slowest request:** use the row with maximum latency from `latency_tokens_cost.csv`; discuss plausible causes only, not a definitive diagnosis.
- **Cost optimization:** keep answers concise with a response token cap; use caching for stable instructions/context; consider a smaller model for simple questions after quality evaluation.
- **Prompt diff:** in MLflow UI, open `eduassist_academic_qa` and compare versions 1, 2, and 3. The prompt text and commit messages are the version evidence.
- **Screenshots to capture manually:** experiment runs and metrics; Prompt Registry showing versions; prompt comparison/diff; `production` alias on the selected version.
- **Report:** populate the report template with actual output from the CSVs. Never claim the code was executed successfully unless you ran it and verified the outputs.